In [2]:
import LinearAlgebra: I, ⋅
import Base.MathConstants: φ
abstract type DescentMethod end

In [3]:
struct GradientDescent <: DescentMethod
  α
end

In [4]:
mutable struct BFGS <: DescentMethod
  Q
end
BFGS(n::Integer) = BFGS(Matrix(1.0I, n, n))

BFGS

In [5]:
mutable struct Momentum <: DescentMethod
  α # learning rate
  β # momentum decay
  v # momentum
end
Momentum(α, β, n::Integer) = Momentum(α, β, zeros(n))

Momentum

In [6]:
mutable struct Adagrad <: DescentMethod
  α # learning rate
  ε # small value
  s # sum of squared gradient
end
Adagrad(α, ε, n::Integer) = Adagrad(α, ε, zeros(n))

Adagrad

In [59]:
mutable struct Adam <: DescentMethod
  α
  β₁
  β₂
  ε
  t
  m
  v
end
Adam(θ; α=0.001, β₁=0.9, β₂=0.999, ε=1e-8) = Adam(α, β₁, β₂, ε, 0, zero(θ), zero(θ))

Adam

In [7]:
mutable struct RMSProp <: DescentMethod
  α # learning rate
  γ # decay
  ε # small value
  s # sum of squared gradient
end
RMSProp(α, γ, ε, n::Integer) = RMSProp(α, γ, ε, zeros(n))

RMSProp

In [8]:
function step!(M::RMSProp, f, ∇f, x)
  α, γ, ε, s, g = M.α, M.γ, M.ε, M.s, ∇f(x)
  s[:] = γ*s + (1-γ)*(g.*g)
  return x - α*g ./ (sqrt.(s) .+ ε)
end

step! (generic function with 1 method)

In [9]:
function step!(M::Adagrad, f, ∇f, x)
  α, ε, s, g = M.α, M.ε, M.s, ∇f(x)
  s[:] += g.*g
  return x - α*g ./ (sqrt.(s) .+ ε)
end

step! (generic function with 2 methods)

In [63]:
function step!(M::Adam, f, ∇f, θ)
  α, β₁, β₂, ε, t = M.α, M.β₁, M.β₂, M.ε, M.t
  m, v, g = M.m, M.v, ∇f(θ)

  m[:] = β₁*m + (1.0 - β₁)*g
  v[:] = β₂*v + (1.0 - β₂)*g.*g

  M.t  = t += 1
  m̂ = m ./ (1.0 - β₁^t)
  v̂ = v ./ (1.0 - β₂^t)
  η = 1.0 # fixed learning rate multiplier

  return θ - η*(α*m̂ ./ (sqrt.(v̂) .+ ε))
end

step! (generic function with 6 methods)

In [10]:
function step!(M::Momentum, f, ∇f, x) 
  α, β, v, g = M.α, M.β, M.v, ∇f(x)
  v[:] = β*v .- α*g
  return x + v
end

step! (generic function with 3 methods)

In [11]:
function step!(M::GradientDescent, f, ∇f, x)
  α, g = M.α, ∇f(x)
  return x - α*g
end

step! (generic function with 4 methods)

In [54]:
function strong_backtracking(f, ∇, x, d; α=1, β=1e-4, σ=0.1)
  y0, g0, y_prev, α_prev = f(x), ∇(x)⋅d, NaN, 0
  αlo, αhi = NaN, NaN
  # bracket phase
  while true
    y = f(x + α*d)
    if y > y0 + β*α*g0 || (!isnan(y_prev) && y ≥ y_prev)
      αlo, αhi = α_prev, α
      break
    end
    g = ∇(x + α*d)⋅d
    if abs(g) ≤ -σ*g0
      return α
    elseif g ≥ 0
      αlo, αhi = α, α_prev
      break
    end
    y_prev, α_prev, α = y, α, 2α
  end
  # zoom phase
  ylo = f(x + αlo*d)
  while true
    α = (αlo + αhi)/2
    y = f(x + α*d)
    if y > y0 + β*α*g0 || y ≥ ylo
      αhi = α
    else
      g = ∇(x + α*d)⋅d
      if abs(g) ≤ -σ*g0
        return α
      elseif g*(αhi - αlo) ≥ 0
        αhi = αlo
      end
      αlo = α
    end
  end
end

function step!(M::BFGS, f, ∇f, x)
  if f(x) ≈ 0.0
    return x
  end

  Q, g = M.Q, ∇f(x)
  α = strong_backtracking(f, ∇f, x, -Q*g)
  x′ = x + α*(-Q*g)
  g′ = ∇f(x′)
  δ = x′ - x
  γ = g′ - g
  Q[:] = Q - (δ*γ'*Q + Q*γ*δ')/(δ'*γ) + (1 + (γ'*Q*γ)/(δ'*γ))[1]*(δ*δ')/(δ'*γ)
  return x′
end

step! (generic function with 5 methods)

In [67]:
function main()
  f(x)  = 100*(x[2] - x[1]^2)^2 + (1-x[1])^2
  ∇f(x) = [400x[1]^3 - 400x[1]*x[2] + 2x[1] - 2,
           200x[2] - 200x[1]^2]
    
  x₀  = [1.1, 2.0]
  pts = [x₀]
  val = Float64[]
  opt = BFGS(2)
  for i=1:25
    push!(val, f(pts[end]))
    push!(pts, step!(opt, f, ∇f, pts[end]))
  end

  pts, val
end

pts, val = main()
val

25-element Array{Float64,1}:
 62.419999999999966
  0.3732411295430105
  0.17993791583456756
  0.06674099904230436
  0.05867991117951793
  0.0024039720448771197
  0.0003657436930539433
  4.534573452787162e-5
  5.897477310015752e-7
  2.8139663178882733e-9
  7.2112375760120156e-12
  2.419852827374445e-16
  2.1203775440114413e-20
  1.2478805770416525e-26
  1.232595164407831e-30
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0